# MLOps with Our Two Spam Models: Student Notebook

In `train_model.ipynb` you trained two spam classifiers and saved them with model cards:

| Model | Trained on | What we learned |
|---|---|---|
| **v1** | `data/sms_real_1k.csv` (85% ham) | High accuracy, but catches almost no spam |
| **v2** | `data/balanced.csv` (50/50) | Catches spam well |

A model in a notebook helps no one (slide 2). In this notebook you take those two models through the rest of the **ML lifecycle**: register them, gate them, serve one, retrain it automatically, roll back a bad version, ship safely and watch for drift.

| Section | What to do |
|---|---|
| **Part A · Starter** | Read the explanations, then fill in every `# TODO`. Run each cell as you go. |
| **Part B · Solution** | Working code and answers. Try Part A first, then compare. |

**Before you start**
1. Run `python make_data.py` once (creates the `data/` files).
2. Run all of `train_model.ipynb` (saves `models/model_nb_v1.pkl`, `models/model_nb_v2.pkl` and their `.json` cards).
   **Stuck on that notebook?** Run `python make_solution_models.py` instead. It creates the same two models and cards from the notebook's solution, so you can start here.

> Everything this notebook creates goes into a separate `mlops_nb/` folder, which it wipes at the start of each run. It never touches `models/production.pkl` or `models/registry.csv` from the command-line lab.

### Why MLOps? (slides 2 and 3)

Training a model is maybe 10% of the work. The other 90% is everything around it: getting it to users, keeping it working, and knowing when it stops working. MLOps borrows tools from three fields to do that:

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart TD
    SE["Software engineering<br/>git, tests, clean code"] --> M["MLOps"]
    DO["DevOps<br/>Docker, CI/CD"] --> M
    ML["Machine learning<br/>data, models, metrics"] --> M
```

What changes when a model leaves the notebook:

| | In a notebook | In production |
|---|---|---|
| Who runs it | You, by pressing Shift+Enter | A server, thousands of times a day |
| Data | One CSV that never changes | New messages every second, and they change over time |
| When it breaks | You see the error | Users see wrong answers, and nobody tells you |
| Which model | "The one in memory" | Must be named, versioned and traceable |
| Updating it | Re-run the cell | Must be tested, gated and rolled out safely |

Every section of this notebook fixes one row of that table.

### Where this fits: the ML lifecycle (slide 5)

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    D["1 Data"] --> T["2 Train"]
    T --> E["3 Evaluate"]
    E --> P["4 Deploy"]
    P --> M["5 Monitor"]
    M -. "problem found" .-> D
```

`train_model.ipynb` covered steps 1 to 3. This notebook covers steps 3 to 5, and the arrow back to 1.

### The map

| Section | Topic | Slides | CLI lab part |
|---|---|---|---|
| 1 | Model registry and accountability | 4, 11 | 2, 4 |
| 2 | Reproducibility check | 4 | 2 |
| 3 | Quality gate and promotion | 7, 11 | 4 |
| 4 | Serving the model as an API | 9, 10 | 3 |
| 5 | Automated retraining pipeline | 11 | 4 |
| 6 | Rollback and data validation | 11 | 4 |
| 7 | Shipping safely: shadow, canary, blue/green | 12 | 5 |
| 8 | Monitoring and drift | 13 | 6 |
| 9 | Retraining after drift, and a fresh test set | 13 | 6 |
| 10 | Closing the loop | 8, 13 | 6 |

### The maturity ladder (slide 8)

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart TD
    L0["Level 0<br/>Manual notebooks"] --> L1["Level 1<br/>Manual hand-off"]
    L1 --> L2["Level 2<br/>Automated training"]
    L2 --> L3["Level 3<br/>Automated deployment"]
    L3 --> L4["Level 4<br/>Monitoring +<br/>auto-retraining"]
```

`train_model.ipynb` was Level 0. By the end of this notebook you will have built, in miniature, every piece of Level 4. What is still missing to make it *real* Level 4 is the plumbing that runs it without you: a scheduler, CI/CD and alerts. We discuss that at the end.

---
# Part A · Starter

Replace each `None` or `...` marked `# TODO` with working code. Hints are in the comments.

Later sections use functions from earlier ones, so work from top to bottom. If you get stuck, copy the matching cell from Part B and keep going.

> **Careful:** the setup cell (A0) empties `mlops_nb/` every time it runs. If you run it again, run every cell after it again too, starting with section 1.

## A0 · Setup

Run these cells as is. They import the libraries, define the helpers you already know from `train_model.ipynb`, and create a clean `mlops_nb/` workspace:

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    W["mlops_nb/"] --> V["versions/<br/>every model ever made"]
    W --> R["registry.csv<br/>one row per version"]
    W --> P["production.pkl<br/>the model users get"]
```

In [ ]:
import hashlib
import json
import os
import shutil
import string
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import train_test_split

FEATURES = ["length", "punct"]
LABEL = "label"

BASE_DATA = "data/balanced.csv"      # the data every retrain starts from
TEST_SET = "data/test_set.csv"       # the fixed test set
WORKDIR = "mlops_nb"
VERSIONS = f"{WORKDIR}/versions"
REGISTRY = f"{WORKDIR}/registry.csv"
PRODUCTION = f"{WORKDIR}/production.pkl"

for f in ["data/balanced.csv", "data/test_set.csv", "data/new_week_ok.csv",
          "models/model_nb_v1.pkl", "models/model_nb_v1.json",
          "models/model_nb_v2.pkl", "models/model_nb_v2.json"]:
    assert os.path.exists(f), (f"{f} is missing. Run train_model.ipynb first, or run `python make_solution_models.py` "
                               "to get the solution models (run make_data.py first if data/ is empty).")

# Start from a clean workspace every time
shutil.rmtree(WORKDIR, ignore_errors=True)
os.makedirs(VERSIONS)
print("Workspace ready:", WORKDIR)

In [ ]:
def evaluate(model, df):
    """Accuracy, precision, recall, F1 (spam = positive class) and the confusion matrix."""
    y_true = df[LABEL]
    y_pred = model.predict(df[FEATURES])
    return {
        "accuracy": round(accuracy_score(y_true, y_pred), 3),
        "precision": round(precision_score(y_true, y_pred, pos_label="spam", zero_division=0), 3),
        "recall": round(recall_score(y_true, y_pred, pos_label="spam", zero_division=0), 3),
        "f1": round(f1_score(y_true, y_pred, pos_label="spam", zero_division=0), 3),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=["ham", "spam"]).tolist(),
    }


def file_fingerprint(path):
    """Short SHA-256 hash of a file. Same file -> same fingerprint."""
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()[:12]


def data_fingerprint(df):
    """Short hash of a DataFrame, for training data that only exists in memory."""
    return hashlib.sha256(pd.util.hash_pandas_object(df, index=False).values.tobytes()).hexdigest()[:12]


def extract_features(text):
    """Turn a raw SMS message into the two numbers our model understands."""
    return {"length": len(text), "punct": sum(1 for ch in text if ch in string.punctuation)}


fixed_test = pd.read_csv(TEST_SET)
print(f"Fixed test set: {len(fixed_test)} messages, {(fixed_test[LABEL] == 'spam').mean():.0%} spam")

In [ ]:
# ---- Registry helpers (run as is) ----
METRICS = ["accuracy", "precision", "recall", "f1"]


def load_registry():
    if not os.path.exists(REGISTRY):
        return pd.DataFrame()
    return pd.read_csv(REGISTRY)


def save_registry(reg):
    reg.to_csv(REGISTRY, index=False)


def register(model, trained_on, fingerprint, metrics, gate_name, gate_result):
    """Save a model under the next version name and add one row to the registry."""
    reg = load_registry()
    version = f"v{len(reg) + 1}"
    joblib.dump(model, f"{VERSIONS}/{version}.pkl")
    row = {
        "version": version,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "trained_on": trained_on,
        "fingerprint": fingerprint,
        **{m: metrics[m] for m in METRICS},
        "gate": gate_name,
        "gate_result": gate_result,     # "pass", "fail" or "not gated"
        "stage": "candidate",
    }
    save_registry(pd.concat([reg, pd.DataFrame([row])], ignore_index=True))
    return version


def load_version(version):
    path = f"{VERSIONS}/{version}.pkl"
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{path} does not exist. Registered so far: {list(load_registry().get('version', []))}. "
            "Run the cells of earlier sections in order (section 1c registers v1 and v2). "
            "Re-running the setup cell empties mlops_nb/, so run everything after it again."
        )
    return joblib.load(path)


def production_version():
    reg = load_registry()
    live = reg.loc[reg["stage"] == "production", "version"]
    return live.iloc[0] if len(live) else None

## A1 · Model registry: know what you have

*Slides 4 and 11.* A **model registry** is the list of every model version you have made, with enough information to answer: *where did it come from, how good was it, and is it live?*

Each version moves through **stages**:

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    C["candidate<br/>just trained"] -- "passes gate" --> P["production<br/>users get it"]
    P -- "replaced" --> A["archived<br/>kept for rollback"]
    A -- "rollback" --> P
```

This is what makes a system **accountable** (slide 4): for any model in production, you can trace it back to its data and its metrics. Real tools like MLflow and SageMaker do exactly this, with a nicer screen.

### 1a · Read the model cards

The cards were written by `train_model.ipynb`.

In [ ]:
cards = {}
for name in ["nb_v1", "nb_v2"]:
    with open(f"models/model_{name}.json") as f:
        cards[name] = json.load(f)

pd.DataFrame(cards).T[["data_file", "data_fingerprint", "rows", "created_at"]]

### 1b · Check the data has not changed

The card stores a **fingerprint** (a hash) of the training file. If we hash the file *today* and get the same value, we know it is the exact same data the model learned from. If it differs, someone changed the file, and the card no longer describes how to rebuild this model.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    F["data file today"] -- "sha256" --> H1["fingerprint now"]
    C["model card"] --> H2["fingerprint then"]
    H1 --> Q{"same?"}
    H2 --> Q
    Q -- "yes" --> OK["traceable"]
    Q -- "no" --> BAD["data changed!"]
```

In [ ]:
for name, card in cards.items():
    # TODO: compute the fingerprint of card["data_file"] as it is today
    # Hint: file_fingerprint(...)
    now = None
    status = "OK, same data" if now == card["data_fingerprint"] else "CHANGED!"
    print(f"{name}: card says {card['data_fingerprint']}, file is now {now} -> {status}")

### 1c · Register both models

We re-check every model on the **fixed test set** instead of trusting the numbers in the card (*trust, but verify*). For now we register them without a gate: the `gate` column says `none`.

In [ ]:
for name, card in cards.items():
    model = joblib.load(f"models/model_{name}.pkl")
    # TODO: evaluate the model on fixed_test
    metrics = None
    # TODO: register it. trained_on = card["data_file"], fingerprint = card["data_fingerprint"],
    # TODO: gate_name = "none", gate_result = "not gated"
    version = None
    print(f"{name} registered as {version}")

assert len(load_registry()) == 2, "The registry should have v1 and v2. Finish the TODOs above and run this cell again."
load_registry()

### 1d · Lineage: follow a model back to its roots

**Lineage** is the chain from a model back to everything that made it. With a registry, answering "where did this model come from?" takes one lookup instead of an afternoon of detective work.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    MOD["model v2"] --> DATA["data file +<br/>fingerprint"]
    MOD --> MET["metrics on<br/>fixed test"]
    MOD --> WHEN["created_at"]
    MOD --> STAGE["stage"]
```

**An analogy:** a used car with a full service history is worth more than one without, even if they drive the same today. When something goes wrong, the history tells you where to look. A registry is the service history of your models.

Run the cell below to print the lineage of each model.

In [ ]:
def lineage(version):
    """Print everything the registry knows about one version."""
    row = load_registry().set_index("version").loc[version]
    print(f"{version}: trained on {row['trained_on']} (fingerprint {row['fingerprint']})")
    print(f"     created {row['created_at']}, stage = {row['stage']}")
    print(f"     accuracy {row['accuracy']:.1%}, precision {row['precision']:.1%}, "
          f"recall {row['recall']:.1%}, F1 {row['f1']:.1%}")


lineage("v1")
lineage("v2")

**Q1.** The registry now has `v1` and `v2`. Six months from now, a user complains that the spam filter is bad. Which columns in the registry would you look at first, and why?

## A2 · Reproducibility: can we rebuild v2?

*Slide 4.* **Reproducible** means: retrain a model months later and get the same result. Let's test that with v2. Rebuild it from scratch using only what the card tells us, then compare it with the saved file.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    CARD["v2 model card"] --> RE["retrain<br/>from scratch"]
    RE --> NEW["rebuilt model"]
    SAVED["model_nb_v2.pkl"] --> CMP{"same?"}
    NEW --> CMP
```

In [ ]:
card = cards["nb_v2"]

# TODO: retrain v2 the way train_model.ipynb did:
# TODO: load card["data_file"], split 75/25 with random_state=42 and stratify on the label,
# TODO: then fit LogisticRegression(max_iter=1000) on the training part
rebuilt = None

saved = joblib.load("models/model_nb_v2.pkl")   # the file train_model.ipynb saved
same_preds = (rebuilt.predict(fixed_test[FEATURES]) == saved.predict(fixed_test[FEATURES])).all()
print("Same predictions on all 500 test messages:", same_preds)
print("Same learned weights:", np.allclose(rebuilt.coef_, saved.coef_) and np.allclose(rebuilt.intercept_, saved.intercept_))

### 2b · What breaks reproducibility?

Getting the *same* model twice is harder than it sounds. Any one of these changes the result:

| Source | Example | How to pin it down |
|---|---|---|
| Randomness | A different `random_state` gives a different train/test split | Fix the seed and record it |
| Data | Someone fixes 3 labels in the CSV | Fingerprint or version the data (DVC) |
| Code | A teammate changes how `punct` is counted | Record the git commit |
| Libraries | scikit-learn 1.3 vs 1.5 use slightly different defaults | Pin versions in `requirements.txt`, use Docker |
| Hardware | GPUs add numbers in a different order | Matters mostly for deep learning |

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    C["code<br/>git commit"] --> R["same model"]
    D["data<br/>fingerprint"] --> R
    P["parameters<br/>seed, sizes"] --> R
    E["environment<br/>library versions"] --> R
```

**Example:** the next cell trains v2 three times on exactly the same data. Only `random_state` differs on the last run.

In [ ]:
df = pd.read_csv(cards["nb_v2"]["data_file"])
for seed in [42, 42, 7]:
    train_part, _ = train_test_split(df, test_size=0.25, random_state=seed, stratify=df[LABEL])
    m = LogisticRegression(max_iter=1000).fit(train_part[FEATURES], train_part[LABEL])
    print(f"random_state={seed:>2}  weights (length, punct) = {np.round(m.coef_[0], 5)}  "
          f"recall on fixed test = {evaluate(m, fixed_test)['recall']:.1%}")

The two runs with `random_state=42` match to the last digit. With `random_state=7` the weights change, even though nothing else did. Here the recall happens to stay at 100%, because this data is easy. With harder data or a bigger model, a different seed can change the metrics too. Either way, without the seed in the model card, you could not prove *which* of these models you shipped.

**Q2.** It worked, but only because you *remembered* `test_size=0.25`, `random_state=42` and `max_iter=1000`. None of them is in the model card. What would you add to the card so that someone else could rebuild the model without asking you?

## A3 · Quality gate: who is allowed into production?

*Slides 7 and 11.* A **quality gate** is an automatic check a model must pass before it can reach users. If it fails, the current production model stays.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    C["candidate"] --> E["evaluate on<br/>fixed test set"]
    E --> G{"gate"}
    G -- "pass" --> P["promote"]
    G -- "fail" --> K["keep current<br/>model"]
```

### 3a · The accuracy-only gate

The repo's first pipeline (`src/mlops_pipeline.py`) used **accuracy > 85%** as its gate. Let's try that gate on v1 and v2.

In [ ]:
ACCURACY_MIN = 0.85


def gate_accuracy_only(metrics):
    # TODO: return True if accuracy is at least ACCURACY_MIN
    return None


reg = load_registry()
reg["accuracy_only_gate"] = [gate_accuracy_only(row) for _, row in reg.iterrows()]
reg[["version", "trained_on", "accuracy", "recall", "accuracy_only_gate"]]

**Q3.** Does v1 pass the accuracy-only gate? Look at its recall. What would happen to users if this gate promoted v1?

### 3b · A better gate

Add a second rule: **spam recall must be at least 0.80**. A model must pass *both* rules.

In [ ]:
RECALL_MIN = 0.80


def gate_strict(metrics):
    # TODO: return True only if accuracy >= ACCURACY_MIN AND recall >= RECALL_MIN
    return None


reg = load_registry()
reg["strict_gate"] = [gate_strict(row) for _, row in reg.iterrows()]
reg[["version", "accuracy", "recall", "strict_gate"]]

### 3c · Kinds of gate checks

Our gate uses **absolute thresholds**: fixed numbers a model must reach. Real pipelines often add more kinds of checks:

| Kind of check | Question it asks | Example rule |
|---|---|---|
| Absolute threshold | Is it good enough? | recall ≥ 0.80 |
| **No regression** (champion vs challenger) | Is it at least as good as what we have? | recall ≥ production's recall − 0.01 |
| Sanity | Does it behave like a spam filter at all? | flags between 5% and 50% of messages |
| Operational | Can we run it? | predicts in under 10 ms, file under 50 MB |

The current production model is called the **champion**, and the new candidate is the **challenger**:

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    CH["challenger<br/>candidate"] --> CMP{"at least as good<br/>on every metric?"}
    CP["champion<br/>production"] --> CMP
    CMP -- "yes" --> W["challenger wins"]
    CMP -- "no" --> L["champion stays"]
```

**Example:** what happens if we pit v1 and v2 against each other with a strict "no metric may drop" rule?

In [ ]:
def no_regression(challenger, champion, tolerance=0.01):
    """Per metric: is the challenger at most `tolerance` worse than the champion?"""
    return {m: challenger[m] >= champion[m] - tolerance for m in METRICS}


reg = load_registry().set_index("version")
pd.DataFrame({
    "v2 challenges v1": no_regression(reg.loc["v2"], reg.loc["v1"]),
    "v1 challenges v2": no_regression(reg.loc["v1"], reg.loc["v2"]),
})

Surprise: **v2 fails the no-regression check against v1**, on precision. v1's precision is 100%, but only because it flagged just 3 messages. A rule that says "no metric may get worse" would have kept the useless model.

The lesson: a gate is only as good as the metrics it watches. Pick the metrics that match what users care about (here, catching spam: recall and F1), and do not blindly require every number to go up.

### 3d · Promote v2

**Promoting** a version means three things happen together:
1. Its file is copied to `production.pkl`, the file the app loads.
2. Its stage becomes `production`.
3. Whatever was in production before becomes `archived`, kept for rollback.

In [ ]:
def promote(version):
    # TODO: copy f"{VERSIONS}/{version}.pkl" to PRODUCTION
    ...
    reg = load_registry()
    # TODO: set stage "archived" on the current production row, then stage "production" on `version`
    # Hint: reg.loc[reg["stage"] == "production", "stage"] = "archived"
    ...
    save_registry(reg)
    print(f"  promoted {version} -> production")


promote("v2")
load_registry()[["version", "trained_on", "recall", "stage"]]

## A4 · Serve it: the model as an API

*Slides 9 and 10.* Apps do not open notebooks. They send a request and get an answer back, like the serverless version in the repo:

> `POST /api/classify {"message": "..."}` → `{"prediction": "spam"}`

```mermaid
sequenceDiagram
    participant App
    participant API as Model API
    participant Log as Prediction log
    App->>API: {"message": "WIN cash!!!"}
    API->>API: extract_features + predict
    API->>Log: features, prediction, version
    API-->>App: {"prediction": "spam"}
```

We will write the API as a plain Python function that takes and returns a dict, the same shape as the JSON. It also writes every request to a **prediction log**. We will need that log for monitoring in section 8.

The API always reports which **model version** answered. When something goes wrong, that is the first thing you need to know.

In [ ]:
prediction_log = []


def classify(request):
    """Pretend API endpoint: POST /api/classify"""
    model = joblib.load(PRODUCTION)   # a real server loads it once, at startup
    # TODO: extract the features of request["message"] and predict with the model
    # Hint: model.predict(pd.DataFrame([features]))[0]
    features, prediction = None, None
    version = production_version()
    prediction_log.append({**features, "prediction": prediction, "model_version": version,
                           "time": datetime.now().isoformat(timespec="seconds")})
    return {"prediction": prediction, "model_version": version}


def health():
    """Pretend API endpoint: GET /api/health"""
    return {"status": "ok", "model_version": production_version()}


print(health())
for msg in ["See you at lunch",
            "URGENT! You have won a 1000 dollar gift card. Call 0800-123-456 now to claim your prize, offer ends today!!! Reply STOP to opt out.",
            "WINNER!! Claim your free prize now!!!"]:
    print(classify({"message": msg}), "<-", msg[:50])

pd.DataFrame(prediction_log)

**From function to real service.** To put this online you would wrap `classify` in a web framework, package it with Docker (slide 9), and pick a host (slide 10):

| Option | You run | Good for | Watch out for |
|---|---|---|---|
| Virtual machine | Everything, on a rented server | First deployments, full control | Always on, always billed |
| Container (Docker, Kubernetes) | An image that runs the same anywhere | Steady traffic, portability | You manage the cluster |
| Serverless (IBM Code Engine, Azure Functions) | Just the function | Spiky traffic, pay per use | Cold starts on the first request |

In this folder, `app.py` and the `Dockerfile` do this with Streamlit: the Dockerfile copies `models/production.pkl` into the image. To serve *this notebook's* production model instead, copy `mlops_nb/production.pkl` over `models/production.pkl` (this replaces the command-line lab's production model), then run `docker build -t spam-lab .` and `docker run -p 8501:8501 spam-lab`.

### 4b · Two ways to serve: online and batch

| | Online (real time) | Batch |
|---|---|---|
| When | One request at a time, as it happens | Many rows at once, on a schedule |
| Speed needed | Milliseconds | Minutes or hours are fine |
| Example | Check each SMS as it arrives | Every night, re-score yesterday's messages |
| Our code | `classify({"message": ...})` | `model.predict(whole_dataframe)` |

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart TD
    subgraph ON["Online"]
        R1["1 request"] --> A1["API"] --> P1["1 answer"]
    end
    subgraph BA["Batch"]
        F["file of<br/>10,000 rows"] --> J["nightly job"] --> O["file of<br/>10,000 answers"]
    end
```

### 4c · Example: why load the model once?

Our `classify` loads `production.pkl` from disk on every request. That keeps the code short, but it is slow. A real server loads the model **once** at startup and keeps it in memory. Let's measure the difference:

In [ ]:
import time

msg = {"message": "WINNER!! Claim your free prize now!!!"}
n = 200

start = time.perf_counter()
for _ in range(n):
    classify(msg)                                   # loads the model every time
reload_ms = (time.perf_counter() - start) / n * 1000
del prediction_log[-n:]                             # keep the log clean

model = joblib.load(PRODUCTION)                     # load once...
start = time.perf_counter()
for _ in range(n):
    model.predict(pd.DataFrame([extract_features(msg["message"])]))   # ...reuse it
loaded_ms = (time.perf_counter() - start) / n * 1000

start = time.perf_counter()
model.predict(fixed_test[FEATURES])                 # batch: all 500 at once
batch_ms = (time.perf_counter() - start) / len(fixed_test) * 1000

print(f"load on every request : {reload_ms:6.2f} ms per message")
print(f"load once             : {loaded_ms:6.2f} ms per message")
print(f"batch of 500          : {batch_ms:6.3f} ms per message")

Your exact numbers depend on your laptop, but the order is always the same: batch is the cheapest per message, and loading the model on every request is the most expensive.

### 4d · Example: bad requests

Real users send broken requests. An API should answer them with a clear error (HTTP status **400 Bad Request**) instead of crashing (status **500**).

In [ ]:
def classify_safe(request):
    message = request.get("message")
    if not isinstance(message, str) or not message.strip():
        return {"status": 400, "error": "'message' must be a non-empty string"}
    return {"status": 200, **classify(request)}


for req in [{"message": "Lunch at 12?"}, {"message": ""}, {"text": "wrong field name"}, {"message": 42}]:
    print(f"{str(req):<32} -> {classify_safe(req)}")

**Q4.** Your API gets 10 requests at 3am and 50,000 at 9am. Which hosting option from the table fits best, and what is the trade-off?

## A5 · An automated retraining pipeline

*Slide 11.* New labeled data arrives every week. Instead of retraining by hand, a **pipeline** does it the same way every time:

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart TD
    I["1 Ingest<br/>base + new"] --> R["2 Retrain"]
    R --> E["3 Evaluate<br/>fixed test set"]
    E --> V["4 Version<br/>save + register"]
    V --> G{"5 Gate"}
    G -- "pass" --> P["promote"]
    G -- "fail" --> K["keep current"]
```

Two design choices to notice:
- **Every** candidate is versioned and registered, even ones the gate rejects. You will see why in section 6.
- The **gate is a parameter**, so we can see what different gates would do.

This is the notebook version of `pipeline.py`.

In [ ]:
def run_pipeline(batch, batch_name, gate_fn, test_df=fixed_test):
    print(f"=== Pipeline: new data = {batch_name}, gate = {gate_fn.__name__} ===")

    # 1 Ingest: our base training data plus the new batch
    # TODO: combine pd.read_csv(BASE_DATA) and batch into one DataFrame
    # Hint: pd.concat([...], ignore_index=True)
    data = None
    print(f"  1 ingest   : {len(data)} rows")

    # 2 Retrain
    # TODO: fit a new LogisticRegression(max_iter=1000) on data
    candidate = None

    # 3 Evaluate on the test set
    # TODO: evaluate the candidate on test_df
    metrics = None
    print(f"  3 evaluate : accuracy {metrics['accuracy']:.1%}, recall {metrics['recall']:.1%}")

    # 4 Version: register it whatever the gate says
    passed = bool(gate_fn(metrics))
    version = register(candidate, trained_on=f"base + {batch_name}", fingerprint=data_fingerprint(data),
                       metrics=metrics, gate_name=gate_fn.__name__, gate_result="pass" if passed else "fail")
    print(f"  4 version  : saved as {version}")

    # 5 Promote only if the gate passes
    if passed:
        # TODO: promote the new version
        ...
    else:
        print(f"  5 gate FAILED -> production stays {production_version()}")
    return version

### What starts the pipeline?

Nobody should have to remember to retrain. A pipeline is started by a **trigger**:

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    T1["schedule<br/>every Monday"] --> P["run_pipeline"]
    T2["new data lands<br/>in storage"] --> P
    T3["code merged<br/>to main"] --> P
    T4["drift alert"] --> P
```

And when a model is promoted, CI/CD takes over to ship it (from `MLOPS_README.md`, section 3):

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart TD
    PR["model promoted"] --> B["rebuild<br/>Docker image"]
    B --> IT["integration<br/>tests"]
    IT --> PU["push image<br/>to registry"]
    PU --> DE["deploy as<br/>shadow or canary"]
```

In this notebook *you* are the trigger and the CI/CD system: you run the cell. Section 10 shows what it looks like when a job decides on its own.

### Why always the same test set?

The pipeline evaluates every candidate on `fixed_test`. If each run used a different random test set, a model could "improve" just because it got an easier test. Same test set → fair comparison between v3, v4, v5 and so on. (Section 9 shows the catch: a fixed test set can get *old*.)

### 5a · A normal week

In [ ]:
week_ok = pd.read_csv("data/new_week_ok.csv")
run_pipeline(week_ok, "new_week_ok", gate_strict)
load_registry()[["version", "trained_on", "accuracy", "recall", "gate", "gate_result", "stage"]]

### 5b · A bad week

An upstream job had a bug: most spam in this batch is labeled "ham". Suppose our pipeline still used the **accuracy-only gate**:

In [ ]:
week_bad = pd.read_csv("data/new_week_bad.csv")
run_pipeline(week_bad, "new_week_bad", gate_accuracy_only)
load_registry()[["version", "trained_on", "accuracy", "recall", "gate", "gate_result", "stage"]]

**Q5.** Which version is in production now? What is its recall? How many of the 75 spam messages in the test set does it catch?

## A6 · Rollback and data validation

*Slide 11: "If something breaks, roll back to yesterday's model."* Because the pipeline **versions every model**, the good one is still in `versions/`. A rollback is just promoting an older version again.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    V3["v3 good<br/>archived"] -- "rollback" --> P["production"]
    V4["v4 bad<br/>production"] -- "archived" --> A["archived"]
```

### 6a · Find the last good version and roll back

In [ ]:
def last_good_version():
    """The newest archived version that passes the strict gate."""
    reg = load_registry()
    # TODO: keep rows where stage == "archived" and gate_strict(row) is True, and return the last version
    # Hint: good = reg[(reg["stage"] == "archived") & reg.apply(gate_strict, axis=1)]
    return None


def rollback():
    target = last_good_version()
    print(f"Rolling back: {production_version()} -> {target}")
    promote(target)


rollback()
load_registry()[["version", "trained_on", "recall", "stage"]]

### 6b · Run the bad week again, with the strict gate

In [ ]:
run_pipeline(week_bad, "new_week_bad", gate_strict)
load_registry()[["version", "trained_on", "accuracy", "recall", "gate", "gate_result", "stage"]]

**Q6.** The strict gate blocked v5. Why do we keep v4 and v5 in the registry at all, if they never should have been used?

### Rollback or roll forward?

When production breaks, there are two ways out:

| | Roll back | Roll forward |
|---|---|---|
| What | Put the previous good version back | Fix the problem and ship a new version |
| Speed | Seconds: the old model is already built | Hours or days: fix, retrain, test |
| Needs | Versioned models (our registry) | A fix you trust |
| Use when | Users are being hurt **now** | The old version is also broken, or the fix is tiny |

Rule of thumb: **roll back first, investigate second.** Stop the damage, then take your time finding out why it happened. The registry tells you where to look: v4's row says `base + new_week_bad`.

### 6c · Catch bad data before training

The gate caught the bad model *after* training. Better still: notice the bad *data* before you train on it. Two cheap checks:

1. **Label mix.** Our weekly batches are usually about 50% spam. Is this one?
2. **Agreement with production.** Our production model is good. If it strongly disagrees with the new labels, the labels are suspicious.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    B["new batch"] --> C1["label mix<br/>as usual?"]
    B --> C2["production agrees<br/>with labels?"]
    C1 --> D{"both OK?"}
    C2 --> D
    D -- "yes" --> T["train"]
    D -- "no" --> S["stop + alert"]
```

In [ ]:
def validate_batch(batch, name):
    prod = joblib.load(PRODUCTION)
    # TODO: compute the share of spam labels in the batch
    spam_share = None
    # TODO: compute the share of rows where the production model's prediction equals the label
    agreement = None
    ok = 0.3 <= spam_share <= 0.7 and agreement >= 0.8
    print(f"{name:<14} spam share {spam_share:5.1%} | production agrees with labels {agreement:5.1%} -> {'OK' if ok else 'SUSPICIOUS'}")
    return {"spam_share": spam_share, "agreement": agreement}


validate_batch(week_ok, "new_week_ok")
validate_batch(week_bad, "new_week_bad")

### 6d · More data checks, with examples

Label checks catch *wrong answers* in the data. **Schema checks** catch *broken data*: missing columns, empty values, impossible numbers. They are cheap, so real pipelines run lots of them.

| Check | Catches | Example of a failure |
|---|---|---|
| Columns | A renamed or missing column | `length` renamed to `len` |
| Missing values | Empty cells | A message whose `punct` was never computed |
| Allowed labels | Typos in labels | `"SPAM"` instead of `"spam"` |
| Value ranges | Impossible numbers | A message with length −3 |
| Label mix | Labeling bugs | 97% ham in a week that is usually 50% |

Run the checks on our weeks, and on a batch we break on purpose:

In [ ]:
def check_schema(batch, name):
    problems = []
    if list(batch.columns) != [LABEL] + FEATURES:
        problems.append(f"columns are {list(batch.columns)}")
    if batch.isna().any().any():
        problems.append(f"{int(batch.isna().sum().sum())} missing values")
    bad_labels = set(batch[LABEL].dropna()) - {"ham", "spam"}
    if bad_labels:
        problems.append(f"unknown labels {bad_labels}")
    if (batch[FEATURES] < 0).any().any():
        problems.append("negative feature values")
    print(f"{name:<18} {'OK' if not problems else 'PROBLEMS: ' + '; '.join(problems)}")


check_schema(week_ok, "new_week_ok")
check_schema(week_bad, "new_week_bad")

broken = week_ok.head(10).copy()
broken.loc[0, "length"] = -3
broken.loc[1, "label"] = "SPAM"
broken.loc[2, "punct"] = None
check_schema(broken, "broken on purpose")

Notice that `new_week_bad` **passes** every schema check. Its rows are perfectly well-formed: the labels are just *wrong*. Schema checks and label checks catch different problems, so you need both.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    B["new batch"] --> S["schema checks<br/>is it well-formed?"]
    S --> L["label checks<br/>is it believable?"]
    L --> T["train"]
```

**Q7.** Look at the two checks for `new_week_bad`. Which one is the stronger signal that the labels are wrong? Where in the pipeline would you put this check?

## A7 · Ship safely: shadow, canary, blue/green

*Slide 12.* Even a model that passes the gate can surprise you on real traffic. So we do not switch every user to it at once (a "big bang"). There are three safer ways.

To test them, we use the 500 messages in `fixed_test` as pretend **live traffic**. Production is the good model, and the candidate is **v4**, the bad model from the bad week. Can each strategy catch it?

In [ ]:
traffic = fixed_test.copy()
prod_model = joblib.load(PRODUCTION)
cand_model = load_version("v4")
print("production:", production_version(), "| candidate: v4 (from new_week_bad)")

### 7a · Shadow deployment

The candidate sees every request, but **users only ever get production's answer**. We compare the two afterwards. Zero risk to users.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    U["request"] --> P["production"]
    U --> C["candidate<br/>shadow"]
    P --> A["answer<br/>to user"]
    C --> L["log only"]
    P --> L
```

Notice that this needs **no labels**. We only compare the two models with each other.

In [ ]:
def shadow(traffic, prod_model, cand_model):
    # TODO: get predictions from both models for every message in traffic
    prod_pred, cand_pred = None, None
    users_get = prod_pred                       # users ONLY see production
    report = {
        "agreement": (prod_pred == cand_pred).mean(),
        "production says spam": (prod_pred == "spam").mean(),
        "candidate says spam": (cand_pred == "spam").mean(),
    }
    return users_get, report


answers, report = shadow(traffic, prod_model, cand_model)
{k: f"{v:.1%}" for k, v in report.items()}

### 7b · Canary deployment

Send a small share of users, here 10%, to the candidate. If its metrics hold, grow the share. If not, only those users were affected.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    U["requests"] --> R{"router"}
    R -- "90%" --> P["production"]
    R -- "10%" --> C["candidate"]
```

In [ ]:
CANARY_SHARE = 0.10
rng = np.random.default_rng(42)
to_canary = rng.random(len(traffic)) < CANARY_SHARE

prod_pred = prod_model.predict(traffic[FEATURES])
cand_pred = cand_model.predict(traffic[FEATURES])

# TODO: users in the canary get cand_pred, everyone else gets prod_pred
# Hint: np.where(to_canary, ..., ...)
answers = None

truth = traffic[LABEL].to_numpy()
summary = pd.DataFrame([
    {"strategy": "shadow", "users on candidate": 0, "wrong answers caused by candidate": 0},
    {"strategy": "canary 10%", "users on candidate": int(to_canary.sum()),
     "wrong answers caused by candidate": int(((answers != truth) & (prod_pred == truth)).sum())},
    {"strategy": "big bang", "users on candidate": len(traffic),
     "wrong answers caused by candidate": int(((cand_pred != truth) & (prod_pred == truth)).sum())},
])
summary

### 7c · Blue/green deployment

Keep **two identical environments**. One is live (blue). Deploy the new model to the idle one (green), test it, then flip the router. Rollback is flipping it back: no retraining, no copying files.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    R{"router"} -- "live" --> B["blue<br/>production"]
    R -. "idle" .-> G["green<br/>candidate"]
```

In [ ]:
router = {"live": "blue", "blue": production_version(), "green": None}


def deploy_to_idle(version):
    idle = "green" if router["live"] == "blue" else "blue"
    router[idle] = version
    print(f"deployed {version} to {idle} (idle)")


def switch():
    # TODO: make the idle environment live
    # Hint: router["live"] = "green" if router["live"] == "blue" else "blue"
    ...
    print(f"router -> {router['live']}, now serving {router[router['live']]}")


deploy_to_idle("v4")
switch()                 # oops, v4 is the bad model...
switch()                 # ...instant rollback
router

### 7d · Example: a canary that ramps up on its own

Real canaries grow in steps (5% → 25% → 50% → 100%), and check a **guardrail metric** at each step. If the guardrail breaks, they roll back automatically.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart TD
    S5["5% of traffic"] --> C1{"guardrail OK?"}
    C1 -- "yes" --> S25["25%"]
    S25 --> C2{"guardrail OK?"}
    C2 -- "yes" --> S100["... 100%"]
    C1 -- "no" --> RB["roll back"]
    C2 -- "no" --> RB
```

Our guardrail needs no labels: *does the candidate flag about the same share of messages as spam as production does, on the same messages?* A gap of more than 5 percentage points stops the rollout.

We try two candidates: the bad **v4**, and **v2**, a good older model.

In [ ]:
def canary_ramp(cand_name, cand_model, prod_model, traffic, steps=(0.05, 0.25, 0.5, 1.0), max_gap=0.05):
    rng = np.random.default_rng(0)
    order = rng.permutation(len(traffic))          # the same users join first at every step
    print(f"--- canary for {cand_name} ---")
    for share in steps:
        users = traffic.iloc[order[: max(1, int(share * len(traffic)))]]
        cand_rate = (cand_model.predict(users[FEATURES]) == "spam").mean()
        prod_rate = (prod_model.predict(users[FEATURES]) == "spam").mean()
        gap = abs(cand_rate - prod_rate)
        print(f"{share:>5.0%} ({len(users):>3} users): candidate flags {cand_rate:5.1%}, "
              f"production {prod_rate:5.1%}, gap {gap:4.1%}")
        if gap > max_gap:
            print(f"      guardrail broken -> ROLL BACK, only {len(users)} users ever saw {cand_name}")
            return False
    print(f"      guardrail held -> {cand_name} now serves everyone")
    return True


v4_survived = canary_ramp("v4", load_version("v4"), prod_model, traffic)
v2_survived = canary_ramp("v2", load_version("v2"), prod_model, traffic)

### The three strategies side by side

| | Shadow | Canary | Blue/green |
|---|---|---|---|
| Users who see the new model | None | A small share, growing | Everyone, after the switch |
| Risk to users | Zero | Small | All at once, but undone in seconds |
| Extra cost | Runs two models on all traffic | Little | Two full environments |
| Finds problems with | Comparing the two models | Guardrails on real users | Tests on the idle environment |
| Rollback | Nothing to roll back | Send the share to 0% | Flip the router |
| Good first step for | A risky new model | A model that passed shadow | Big releases, instant undo |

They combine well: **shadow first** to compare safely, then a **canary** to meet real users, with **blue/green** environments so any rollback is instant.

**Q8.** Suppose the bad model v4 had slipped past every gate. Which strategy finds the problem with **zero** users affected? Which one gives the **fastest** rollback? Use the numbers above.

## A8 · Monitoring: watch for drift

*Slide 13.* A model is trained on the past but used in the future. Two things can change:

| Kind | What changes | Spam example | Can you see it without labels? |
|---|---|---|---|
| **Data drift** | The inputs | Spammers start sending shorter messages | **Yes**: compare feature averages |
| **Concept drift** | What the label means | "Your package is waiting" used to be ham, now it is mostly scams | No: you need fresh labels to see accuracy drop |

In production, labels arrive late or never, so we monitor what we *can* see right away:
- **Input drift:** the average `length` and `punct` of new messages, compared with the training data.
- **Prediction drift:** the share of messages the model calls spam. If it suddenly drops, something changed.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    REF["reference<br/>training data"] --> CMP{"compare"}
    NEW["this week's<br/>messages"] --> CMP
    CMP -- "change > 25%" --> AL["DRIFT alert"]
    CMP -- "small change" --> OK["ok"]
```

Unlike `check_drift.py`, which only looks at the spam messages, this report uses **all** messages and **no labels**, like a real monitor would.

In [ ]:
reference = pd.read_csv(BASE_DATA)
DRIFT_THRESHOLD = 0.25   # a change of more than 25% counts as drift


def drift_report(batch, model):
    rows = []
    for feature in FEATURES:
        # TODO: compare the average of `feature` in reference and in batch
        before, after = None, None
        rows.append({"check": f"average {feature}", "reference": before, "this week": after})
    # TODO: compare the share of messages the model predicts as spam, in reference and in batch
    before, after = None, None
    rows.append({"check": "share predicted spam", "reference": before, "this week": after})

    report = pd.DataFrame(rows)
    report["change"] = (report["this week"] - report["reference"]) / report["reference"]
    report["drift"] = report["change"].abs() > DRIFT_THRESHOLD
    return report.round(3)


week_drift = pd.read_csv("data/new_week_drift.csv")
prod_model = joblib.load(PRODUCTION)
print("new_week_ok:")
display(drift_report(week_ok, prod_model))
print("new_week_drift:")
display(drift_report(week_drift, prod_model))

### 8b · See the drift

Averages hide a lot. Here is the whole distribution of message length, as text bars, for the training data and for the drift week:

In [ ]:
bins = [0, 25, 50, 75, 100, 150, 200, 400]
ref_share = pd.cut(reference["length"], bins).value_counts(normalize=True).sort_index()
new_share = pd.cut(week_drift["length"], bins).value_counts(normalize=True).sort_index()

print(f"{'length':>12}   {'training data':<26} {'drift week':<26}")
for interval in ref_share.index:
    r, n = ref_share[interval], new_share[interval]
    print(f"{str(interval):>12}   {'█' * round(r * 40):<20}{r:4.0%}   {'█' * round(n * 40):<20}{n:4.0%}")

The training data has two humps: short ham and long spam. In the drift week almost everything is short, because spam moved into ham's territory. A model that separates messages by length cannot tell them apart any more.

### Drift happens everywhere

| Domain | Data drift (inputs change) | Concept drift (meaning changes) |
|---|---|---|
| Spam | Spammers send shorter messages | "Your package is waiting" turns from ham into scam |
| Online shopping | A pandemic changes what people buy | "Normal" basket size changes for good |
| Fraud | A new payment app brings new kinds of users | Criminals invent a new trick |
| Medical images | The hospital buys a new scanner | A new disease variant looks different |

### What to watch: four layers of monitoring

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    SYS["System<br/>latency, errors, uptime"] --> DASH["dashboard<br/>+ alerts"]
    DAT["Data<br/>feature averages, missing values"] --> DASH
    MOD["Model<br/>share predicted spam"] --> DASH
    BUS["Business<br/>user spam reports"] --> DASH
```

Our `drift_report` covers the middle two. The system layer is what DevOps already monitors. The business layer, such as users pressing "report spam", is often the **first** place a broken model shows up, and it gives you free labels.

**Choosing the threshold.** We use 25%. Too low, and normal week-to-week noise sets off alarms nobody reads ("alert fatigue"). Too high, and you notice drift weeks late. Look at the normal week above: its biggest change was about 8%, so 25% leaves room for noise.

### 8c · When the labels arrive

A few days later, someone labels the drifted batch. Now we can measure how production *really* did on it.

In [ ]:
pd.DataFrame({
    "normal week": evaluate(prod_model, week_ok),
    "drift week": evaluate(prod_model, week_drift),
}).T[METRICS]

**Q9.** Which signals in the drift report fired for `new_week_drift`? Which one would you trust most as an early alarm, and why? How bad is production's recall once the labels arrive?

## A9 · Retrain after drift, and refresh the test set

The obvious fix is to retrain with the new kind of spam. We split the labeled drift batch in two: one half to **train** on, and one half kept back to **test** with.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    D["labeled drift<br/>batch"] --> S{"split 50/50"}
    S --> TR["drift_train<br/>for the pipeline"]
    S --> TE["drift_test<br/>new kind of spam"]
```

In [ ]:
drift_train, drift_test = train_test_split(week_drift, test_size=0.5, random_state=42, stratify=week_drift[LABEL])
new_version = run_pipeline(drift_train, "drift_train", gate_strict)

The gate passed. But the gate only looked at `fixed_test`, which contains only the **old** kind of spam. How does the new model do on the *new* kind?

In [ ]:
new_model = load_version(new_version)
# TODO: evaluate new_model on fixed_test and on drift_test, and show both in one table
...

### 9b · Refresh the test set

A test set must look like **today's** traffic. Add the held-back drift messages to it, and run the gate again on the production model.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    OLD["fixed_test<br/>500 old"] --> NEWT["refreshed_test"]
    DT["drift_test<br/>250 new"] --> NEWT
    NEWT --> G{"strict gate"}
```

In [ ]:
# TODO: build refreshed_test by stacking fixed_test and drift_test
refreshed_test = None

prod_model = joblib.load(PRODUCTION)
# TODO: evaluate the production model on refreshed_test and check it with gate_strict
m = None
print(f"production = {production_version()} on refreshed test: accuracy {m['accuracy']:.1%}, recall {m['recall']:.1%}")
print("passes strict gate on the refreshed test set:", gate_strict(m))

### 9c · Keeping the test set fresh

A test set is a snapshot of the world on the day you made it. Common ways to keep it current:

| Policy | How it works | Trade-off |
|---|---|---|
| Fixed | Never changes | Fair comparisons over time, but gets old (our problem) |
| Rolling window | Always the last N weeks of labeled data | Always current, but older models' scores can't be compared |
| Fixed core + fresh slice | A stable core plus recent examples (what we just did) | A balance of both |

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    W1["week 1"] --> TS["test set"]
    W2["week 2"] --> TS
    W3["week 3"] --> TS
    W4["new week"] -- "add" --> TS
    W0["oldest week"] -. "drop" .-> TS
```

**Example:** how much of the refreshed test set is the new kind of spam?

In [ ]:
mix = pd.DataFrame({
    "messages": [len(fixed_test), len(drift_test)],
    "spam": [(fixed_test[LABEL] == "spam").sum(), (drift_test[LABEL] == "spam").sum()],
}, index=["old (fixed_test)", "new (drift_test)"])
mix["share of all spam"] = (mix["spam"] / mix["spam"].sum()).round(2)
mix

**Q10.** The new model passed the gate on the old test set but fails it on the refreshed one. Give two reasons why retraining did not fix the problem. What would actually fix it?

## A10 · Close the loop

*Slides 8 and 13: "drift crosses a threshold → alert → retrain → gate → redeploy."* Let's put the pieces together into one **weekly job** that a scheduler could run every Monday:

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart TD
    W["new batch"] --> M["monitor:<br/>drift_report"]
    M -- "no drift" --> N["nothing to do"]
    M -- "drift" --> V["validate_batch"]
    V -- "label mix OK" --> P["run_pipeline<br/>strict gate,<br/>refreshed test"]
    V -- "label mix wrong" --> H1["stop:<br/>check labels"]
    P -- "pass" --> D["promote"]
    P -- "fail" --> H["alert a human"]
```

One subtlety: in section 6c, `validate_batch` also checked whether production **agrees** with the new labels. During drift, production is *expected* to disagree: that is what drift means. So the weekly job only stops on a wrong **label mix**, and lets the gate judge the rest.

We replay two weeks through the job: the normal week, and the drift week's training half from section 9.

In [ ]:
def weekly_job(batch, name):
    print(f"\n##### Weekly job: {name} #####")
    report = drift_report(batch, joblib.load(PRODUCTION))
    if not report["drift"].any():
        print("No drift. Nothing to do.")
        return
    print("ALERT: drift in", ", ".join(report.loc[report["drift"], "check"]))
    check = validate_batch(batch, name)
    if not 0.3 <= check["spam_share"] <= 0.7:
        print("STOP: the label mix looks wrong. A human needs to check the labels.")
        return
    print("Label mix OK. Production disagreeing with the labels is expected during drift: the gate decides.")
    # TODO: run the pipeline on the batch with gate_strict and the refreshed test set
    version = None
    if production_version() != version:
        print("Gate failed: a human needs to look at this (better features? more labels?)")


weekly_job(week_ok, "new_week_ok")
weekly_job(drift_train, "drift_train")

### What the job did, week by week

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart TD
    W1["normal week"] --> N1["no drift<br/>no action"]
    W2["drift week"] --> A2["drift alert"]
    A2 --> V2["label mix OK"]
    V2 --> P2["pipeline on<br/>fresh test set"]
    P2 --> F2["gate fails<br/>human alerted"]
```

That last box is a good outcome. Automation handled the routine week by itself, and passed the hard week to a person with a clear explanation. Good MLOps does not mean "no humans". It means humans only spend time where they are needed.

### The full history

The registry now tells the whole story of this notebook, one row per model.

In [ ]:
load_registry()[["version", "trained_on", "accuracy", "recall", "gate", "gate_result", "stage"]]

**Q11.** Read the registry from top to bottom and tell the story in one sentence per version. Which version is in production at the end, and why is that the right call?

## Wrap-up: the four goals (slide 4)

| Goal | Where you saw it in this notebook |
|---|---|
| **Reproducible** | Section 2: you rebuilt v2 and got identical weights |
| **Accountable** | Sections 1 and 6: fingerprints and the registry trace every model to its data |
| **Collaborative** | The registry and model cards let *someone else* understand and roll back your models |
| **Continuous** | Sections 5, 8 and 10: the pipeline, the monitor and the weekly job run without you choosing what to do |

### What is still missing for real Level 4?

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    S["scheduler<br/>runs weekly_job"] --> J["weekly_job"]
    J --> CI["CI/CD<br/>rebuild Docker image"]
    CI --> RO["canary or<br/>shadow rollout"]
    J --> AL["alerts<br/>Slack, PagerDuty"]
    DV["DVC<br/>data versions"] --> J
```

| Piece | Tool examples | Replaces in this notebook |
|---|---|---|
| Scheduler | cron, Airflow, GitHub Actions | You pressing Shift+Enter |
| CI/CD | GitHub Actions, Jenkins | Copying `production.pkl` and rebuilding Docker by hand |
| Data versioning | DVC, Delta Lake | Our 12-character fingerprints |
| Model registry | MLflow, SageMaker, Vertex AI | `registry.csv` |
| Alerts | Slack, PagerDuty | `print("ALERT ...")` |
| Explainability | SHAP, LIME | Looking at `length` and `punct` by eye |

**Q12.** Pick **one** piece from the table above. In two or three sentences, say how you would add it to this project and which problem from this notebook it would have caught sooner.

## Cheat sheet

| Term | One-line meaning |
|---|---|
| Model registry | The list of every model version, its data, metrics and stage |
| Stage | candidate → production → archived |
| Quality gate | Automatic rules a model must pass before production |
| Promote / rollback | Make a version live / make an older version live again |
| Data validation | Checking a new batch looks sane *before* training on it |
| Shadow | Candidate sees traffic, users only get production's answers |
| Canary | A small share of users get the candidate |
| Blue/green | Two environments; flip the router to switch or roll back |
| Data drift | The inputs change |
| Concept drift | What the label means changes |
| Prediction drift | The share of each prediction changes; needs no labels |

---
# Part B · Solution

This section runs on its own from top to bottom. The explanations are in Part A; here you get the code and the answers.

## B0 · Setup

In [ ]:
import hashlib
import json
import os
import shutil
import string
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import train_test_split

FEATURES = ["length", "punct"]
LABEL = "label"

BASE_DATA = "data/balanced.csv"      # the data every retrain starts from
TEST_SET = "data/test_set.csv"       # the fixed test set
WORKDIR = "mlops_nb"
VERSIONS = f"{WORKDIR}/versions"
REGISTRY = f"{WORKDIR}/registry.csv"
PRODUCTION = f"{WORKDIR}/production.pkl"

for f in ["data/balanced.csv", "data/test_set.csv", "data/new_week_ok.csv",
          "models/model_nb_v1.pkl", "models/model_nb_v1.json",
          "models/model_nb_v2.pkl", "models/model_nb_v2.json"]:
    assert os.path.exists(f), (f"{f} is missing. Run train_model.ipynb first, or run `python make_solution_models.py` "
                               "to get the solution models (run make_data.py first if data/ is empty).")

# Start from a clean workspace every time
shutil.rmtree(WORKDIR, ignore_errors=True)
os.makedirs(VERSIONS)
print("Workspace ready:", WORKDIR)

In [ ]:
def evaluate(model, df):
    """Accuracy, precision, recall, F1 (spam = positive class) and the confusion matrix."""
    y_true = df[LABEL]
    y_pred = model.predict(df[FEATURES])
    return {
        "accuracy": round(accuracy_score(y_true, y_pred), 3),
        "precision": round(precision_score(y_true, y_pred, pos_label="spam", zero_division=0), 3),
        "recall": round(recall_score(y_true, y_pred, pos_label="spam", zero_division=0), 3),
        "f1": round(f1_score(y_true, y_pred, pos_label="spam", zero_division=0), 3),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=["ham", "spam"]).tolist(),
    }


def file_fingerprint(path):
    """Short SHA-256 hash of a file. Same file -> same fingerprint."""
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()[:12]


def data_fingerprint(df):
    """Short hash of a DataFrame, for training data that only exists in memory."""
    return hashlib.sha256(pd.util.hash_pandas_object(df, index=False).values.tobytes()).hexdigest()[:12]


def extract_features(text):
    """Turn a raw SMS message into the two numbers our model understands."""
    return {"length": len(text), "punct": sum(1 for ch in text if ch in string.punctuation)}


fixed_test = pd.read_csv(TEST_SET)
print(f"Fixed test set: {len(fixed_test)} messages, {(fixed_test[LABEL] == 'spam').mean():.0%} spam")

In [ ]:
# ---- Registry helpers (run as is) ----
METRICS = ["accuracy", "precision", "recall", "f1"]


def load_registry():
    if not os.path.exists(REGISTRY):
        return pd.DataFrame()
    return pd.read_csv(REGISTRY)


def save_registry(reg):
    reg.to_csv(REGISTRY, index=False)


def register(model, trained_on, fingerprint, metrics, gate_name, gate_result):
    """Save a model under the next version name and add one row to the registry."""
    reg = load_registry()
    version = f"v{len(reg) + 1}"
    joblib.dump(model, f"{VERSIONS}/{version}.pkl")
    row = {
        "version": version,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "trained_on": trained_on,
        "fingerprint": fingerprint,
        **{m: metrics[m] for m in METRICS},
        "gate": gate_name,
        "gate_result": gate_result,     # "pass", "fail" or "not gated"
        "stage": "candidate",
    }
    save_registry(pd.concat([reg, pd.DataFrame([row])], ignore_index=True))
    return version


def load_version(version):
    path = f"{VERSIONS}/{version}.pkl"
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{path} does not exist. Registered so far: {list(load_registry().get('version', []))}. "
            "Run the cells of earlier sections in order (section 1c registers v1 and v2). "
            "Re-running the setup cell empties mlops_nb/, so run everything after it again."
        )
    return joblib.load(path)


def production_version():
    reg = load_registry()
    live = reg.loc[reg["stage"] == "production", "version"]
    return live.iloc[0] if len(live) else None

## B1 · Model registry: know what you have

In [ ]:
cards = {}
for name in ["nb_v1", "nb_v2"]:
    with open(f"models/model_{name}.json") as f:
        cards[name] = json.load(f)

pd.DataFrame(cards).T[["data_file", "data_fingerprint", "rows", "created_at"]]

### 1b · Check the data has not changed

In [ ]:
for name, card in cards.items():
    now = file_fingerprint(card["data_file"])
    status = "OK, same data" if now == card["data_fingerprint"] else "CHANGED!"
    print(f"{name}: card says {card['data_fingerprint']}, file is now {now} -> {status}")

### 1c · Register both models

In [ ]:
for name, card in cards.items():
    model = joblib.load(f"models/model_{name}.pkl")
    metrics = evaluate(model, fixed_test)
    version = register(model, trained_on=card["data_file"], fingerprint=card["data_fingerprint"],
                       metrics=metrics, gate_name="none", gate_result="not gated")
    print(f"{name} registered as {version}")

assert len(load_registry()) == 2, "The registry should have v1 and v2. Finish the TODOs above and run this cell again."
load_registry()

### 1d · Lineage

In [ ]:
def lineage(version):
    """Print everything the registry knows about one version."""
    row = load_registry().set_index("version").loc[version]
    print(f"{version}: trained on {row['trained_on']} (fingerprint {row['fingerprint']})")
    print(f"     created {row['created_at']}, stage = {row['stage']}")
    print(f"     accuracy {row['accuracy']:.1%}, precision {row['precision']:.1%}, "
          f"recall {row['recall']:.1%}, F1 {row['f1']:.1%}")


lineage("v1")
lineage("v2")

**A1.** `stage` tells you which version is live. Then `trained_on` and `fingerprint` tell you exactly which data produced it, and the metric columns tell you how good it was *when it was released*. Comparing those numbers with how it performs today tells you whether the model got worse or the world changed. This is what slide 4 means by **accountable**.

## B2 · Reproducibility: can we rebuild v2?

In [ ]:
card = cards["nb_v2"]

df = pd.read_csv(card["data_file"])
train_part, _ = train_test_split(df, test_size=0.25, random_state=42, stratify=df[LABEL])
rebuilt = LogisticRegression(max_iter=1000).fit(train_part[FEATURES], train_part[LABEL])

saved = joblib.load("models/model_nb_v2.pkl")   # the file train_model.ipynb saved
same_preds = (rebuilt.predict(fixed_test[FEATURES]) == saved.predict(fixed_test[FEATURES])).all()
print("Same predictions on all 500 test messages:", same_preds)
print("Same learned weights:", np.allclose(rebuilt.coef_, saved.coef_) and np.allclose(rebuilt.intercept_, saved.intercept_))

### 2b · What breaks reproducibility?

In [ ]:
df = pd.read_csv(cards["nb_v2"]["data_file"])
for seed in [42, 42, 7]:
    train_part, _ = train_test_split(df, test_size=0.25, random_state=seed, stratify=df[LABEL])
    m = LogisticRegression(max_iter=1000).fit(train_part[FEATURES], train_part[LABEL])
    print(f"random_state={seed:>2}  weights (length, punct) = {np.round(m.coef_[0], 5)}  "
          f"recall on fixed test = {evaluate(m, fixed_test)['recall']:.1%}")

**A2.** The **training parameters** (`test_size`, `random_state`, `stratify`, `max_iter`), the **library versions** (scikit-learn, pandas, Python), and the **code version** (the git commit of the training script). With the data fingerprint, that is everything needed to rebuild the model exactly. Tools like MLflow record these automatically for every run.

## B3 · Quality gate: who is allowed into production?

In [ ]:
ACCURACY_MIN = 0.85


def gate_accuracy_only(metrics):
    return metrics["accuracy"] >= ACCURACY_MIN


reg = load_registry()
reg["accuracy_only_gate"] = [gate_accuracy_only(row) for _, row in reg.iterrows()]
reg[["version", "trained_on", "accuracy", "recall", "accuracy_only_gate"]]

**A3.** Yes. v1 has 85.6% accuracy, just above 85%, so it **passes**. But its recall is 4%: it catches 3 of 75 spam messages. Users would get almost all their spam. This is exactly slide 7's warning, now happening inside an automated system: *an automated quality gate that only checks accuracy would happily promote* a useless model.

### 3b · A better gate

In [ ]:
RECALL_MIN = 0.80


def gate_strict(metrics):
    return metrics["accuracy"] >= ACCURACY_MIN and metrics["recall"] >= RECALL_MIN


reg = load_registry()
reg["strict_gate"] = [gate_strict(row) for _, row in reg.iterrows()]
reg[["version", "accuracy", "recall", "strict_gate"]]

### 3c · Kinds of gate checks

In [ ]:
def no_regression(challenger, champion, tolerance=0.01):
    """Per metric: is the challenger at most `tolerance` worse than the champion?"""
    return {m: challenger[m] >= champion[m] - tolerance for m in METRICS}


reg = load_registry().set_index("version")
pd.DataFrame({
    "v2 challenges v1": no_regression(reg.loc["v2"], reg.loc["v1"]),
    "v1 challenges v2": no_regression(reg.loc["v1"], reg.loc["v2"]),
})

v2 fails on precision, because v1's 100% precision came from flagging only 3 messages. A gate has to watch the metrics users care about.

### 3d · Promote v2

In [ ]:
def promote(version):
    shutil.copy(f"{VERSIONS}/{version}.pkl", PRODUCTION)
    reg = load_registry()
    reg.loc[reg["stage"] == "production", "stage"] = "archived"
    reg.loc[reg["version"] == version, "stage"] = "production"
    save_registry(reg)
    print(f"  promoted {version} -> production")


promote("v2")
load_registry()[["version", "trained_on", "recall", "stage"]]

## B4 · Serve it: the model as an API

In [ ]:
prediction_log = []


def classify(request):
    """Pretend API endpoint: POST /api/classify"""
    model = joblib.load(PRODUCTION)   # a real server loads it once, at startup
    features = extract_features(request["message"])
    prediction = model.predict(pd.DataFrame([features]))[0]
    version = production_version()
    prediction_log.append({**features, "prediction": prediction, "model_version": version,
                           "time": datetime.now().isoformat(timespec="seconds")})
    return {"prediction": prediction, "model_version": version}


def health():
    """Pretend API endpoint: GET /api/health"""
    return {"status": "ok", "model_version": production_version()}


print(health())
for msg in ["See you at lunch",
            "URGENT! You have won a 1000 dollar gift card. Call 0800-123-456 now to claim your prize, offer ends today!!! Reply STOP to opt out.",
            "WINNER!! Claim your free prize now!!!"]:
    print(classify({"message": msg}), "<-", msg[:50])

pd.DataFrame(prediction_log)

### 4b · Online vs batch, and loading the model once

In [ ]:
import time

msg = {"message": "WINNER!! Claim your free prize now!!!"}
n = 200

start = time.perf_counter()
for _ in range(n):
    classify(msg)                                   # loads the model every time
reload_ms = (time.perf_counter() - start) / n * 1000
del prediction_log[-n:]                             # keep the log clean

model = joblib.load(PRODUCTION)                     # load once...
start = time.perf_counter()
for _ in range(n):
    model.predict(pd.DataFrame([extract_features(msg["message"])]))   # ...reuse it
loaded_ms = (time.perf_counter() - start) / n * 1000

start = time.perf_counter()
model.predict(fixed_test[FEATURES])                 # batch: all 500 at once
batch_ms = (time.perf_counter() - start) / len(fixed_test) * 1000

print(f"load on every request : {reload_ms:6.2f} ms per message")
print(f"load once             : {loaded_ms:6.2f} ms per message")
print(f"batch of 500          : {batch_ms:6.3f} ms per message")

### 4d · Bad requests

In [ ]:
def classify_safe(request):
    message = request.get("message")
    if not isinstance(message, str) or not message.strip():
        return {"status": 400, "error": "'message' must be a non-empty string"}
    return {"status": 200, **classify(request)}


for req in [{"message": "Lunch at 12?"}, {"message": ""}, {"text": "wrong field name"}, {"message": 42}]:
    print(f"{str(req):<32} -> {classify_safe(req)}")

**A4.** **Serverless.** It scales up for the 9am spike and down to zero at night, so you pay almost nothing while idle. The trade-off is **cold starts**: the first request after a quiet period is slow while the function starts and loads the model. A VM would be fast all the time, but you would pay for the 9am capacity all night.

## B5 · An automated retraining pipeline

In [ ]:
def run_pipeline(batch, batch_name, gate_fn, test_df=fixed_test):
    print(f"=== Pipeline: new data = {batch_name}, gate = {gate_fn.__name__} ===")

    # 1 Ingest: our base training data plus the new batch
    data = pd.concat([pd.read_csv(BASE_DATA), batch], ignore_index=True)
    print(f"  1 ingest   : {len(data)} rows")

    # 2 Retrain
    candidate = LogisticRegression(max_iter=1000).fit(data[FEATURES], data[LABEL])

    # 3 Evaluate on the test set
    metrics = evaluate(candidate, test_df)
    print(f"  3 evaluate : accuracy {metrics['accuracy']:.1%}, recall {metrics['recall']:.1%}")

    # 4 Version: register it whatever the gate says
    passed = bool(gate_fn(metrics))
    version = register(candidate, trained_on=f"base + {batch_name}", fingerprint=data_fingerprint(data),
                       metrics=metrics, gate_name=gate_fn.__name__, gate_result="pass" if passed else "fail")
    print(f"  4 version  : saved as {version}")

    # 5 Promote only if the gate passes
    if passed:
        promote(version)
    else:
        print(f"  5 gate FAILED -> production stays {production_version()}")
    return version

### 5a · A normal week

In [ ]:
week_ok = pd.read_csv("data/new_week_ok.csv")
run_pipeline(week_ok, "new_week_ok", gate_strict)
load_registry()[["version", "trained_on", "accuracy", "recall", "gate", "gate_result", "stage"]]

### 5b · A bad week

In [ ]:
week_bad = pd.read_csv("data/new_week_bad.csv")
run_pipeline(week_bad, "new_week_bad", gate_accuracy_only)
load_registry()[["version", "trained_on", "accuracy", "recall", "gate", "gate_result", "stage"]]

**A5.** **v4**, trained on the bad batch, is in production. Its accuracy is 86.8%, so the accuracy-only gate let it through, but its recall is 12%: it catches only **9 of 75** spam messages. One bad data batch plus a weak gate, and a broken model reached users automatically. Automation makes mistakes faster, not just good work.

## B6 · Rollback and data validation

In [ ]:
def last_good_version():
    """The newest archived version that passes the strict gate."""
    reg = load_registry()
    good = reg[(reg["stage"] == "archived") & reg.apply(gate_strict, axis=1)]
    return good["version"].iloc[-1]


def rollback():
    target = last_good_version()
    print(f"Rolling back: {production_version()} -> {target}")
    promote(target)


rollback()
load_registry()[["version", "trained_on", "recall", "stage"]]

### 6b · Run the bad week again, with the strict gate

In [ ]:
run_pipeline(week_bad, "new_week_bad", gate_strict)
load_registry()[["version", "trained_on", "accuracy", "recall", "gate", "gate_result", "stage"]]

**A6.** Three reasons. **Rollback:** archived versions are what make rollback possible (v3 came back that way). **Investigation:** v4 and v5 are evidence. Both rows point to `new_week_bad`, which tells you the data batch was the problem, not the code. **Comparison:** you can see how quality changes over time. Deleting rejected models would destroy the audit trail that makes the system accountable.

### 6c · Catch bad data before training

In [ ]:
def validate_batch(batch, name):
    prod = joblib.load(PRODUCTION)
    spam_share = (batch[LABEL] == "spam").mean()
    agreement = (prod.predict(batch[FEATURES]) == batch[LABEL]).mean()
    ok = 0.3 <= spam_share <= 0.7 and agreement >= 0.8
    print(f"{name:<14} spam share {spam_share:5.1%} | production agrees with labels {agreement:5.1%} -> {'OK' if ok else 'SUSPICIOUS'}")
    return {"spam_share": spam_share, "agreement": agreement}


validate_batch(week_ok, "new_week_ok")
validate_batch(week_bad, "new_week_bad")

### 6d · Schema checks

In [ ]:
def check_schema(batch, name):
    problems = []
    if list(batch.columns) != [LABEL] + FEATURES:
        problems.append(f"columns are {list(batch.columns)}")
    if batch.isna().any().any():
        problems.append(f"{int(batch.isna().sum().sum())} missing values")
    bad_labels = set(batch[LABEL].dropna()) - {"ham", "spam"}
    if bad_labels:
        problems.append(f"unknown labels {bad_labels}")
    if (batch[FEATURES] < 0).any().any():
        problems.append("negative feature values")
    print(f"{name:<18} {'OK' if not problems else 'PROBLEMS: ' + '; '.join(problems)}")


check_schema(week_ok, "new_week_ok")
check_schema(week_bad, "new_week_bad")

broken = week_ok.head(10).copy()
broken.loc[0, "length"] = -3
broken.loc[1, "label"] = "SPAM"
broken.loc[2, "punct"] = None
check_schema(broken, "broken on purpose")

**A7.** Agreement with production is the stronger signal. On the bad batch, the production model agrees with only about 17% of the labels, compared with about 99% on a normal week. A good model and a dataset disagreeing that badly means one of them is broken, and the model passed its gate last week. The label mix (97% ham instead of about 50%) is also a red flag. Put the check **between step 1 (ingest) and step 2 (retrain)**. Then a bad batch never even produces a candidate, and someone is alerted to fix the upstream bug.

## B7 · Ship safely: shadow, canary, blue/green

In [ ]:
traffic = fixed_test.copy()
prod_model = joblib.load(PRODUCTION)
cand_model = load_version("v4")
print("production:", production_version(), "| candidate: v4 (from new_week_bad)")

### 7a · Shadow deployment

In [ ]:
def shadow(traffic, prod_model, cand_model):
    prod_pred = prod_model.predict(traffic[FEATURES])
    cand_pred = cand_model.predict(traffic[FEATURES])
    users_get = prod_pred                       # users ONLY see production
    report = {
        "agreement": (prod_pred == cand_pred).mean(),
        "production says spam": (prod_pred == "spam").mean(),
        "candidate says spam": (cand_pred == "spam").mean(),
    }
    return users_get, report


answers, report = shadow(traffic, prod_model, cand_model)
{k: f"{v:.1%}" for k, v in report.items()}

### 7b · Canary deployment

In [ ]:
CANARY_SHARE = 0.10
rng = np.random.default_rng(42)
to_canary = rng.random(len(traffic)) < CANARY_SHARE

prod_pred = prod_model.predict(traffic[FEATURES])
cand_pred = cand_model.predict(traffic[FEATURES])

answers = np.where(to_canary, cand_pred, prod_pred)

truth = traffic[LABEL].to_numpy()
summary = pd.DataFrame([
    {"strategy": "shadow", "users on candidate": 0, "wrong answers caused by candidate": 0},
    {"strategy": "canary 10%", "users on candidate": int(to_canary.sum()),
     "wrong answers caused by candidate": int(((answers != truth) & (prod_pred == truth)).sum())},
    {"strategy": "big bang", "users on candidate": len(traffic),
     "wrong answers caused by candidate": int(((cand_pred != truth) & (prod_pred == truth)).sum())},
])
summary

### 7c · Blue/green deployment

In [ ]:
router = {"live": "blue", "blue": production_version(), "green": None}


def deploy_to_idle(version):
    idle = "green" if router["live"] == "blue" else "blue"
    router[idle] = version
    print(f"deployed {version} to {idle} (idle)")


def switch():
    router["live"] = "green" if router["live"] == "blue" else "blue"
    print(f"router -> {router['live']}, now serving {router[router['live']]}")


deploy_to_idle("v4")
switch()                 # oops, v4 is the bad model...
switch()                 # ...instant rollback
router

### 7d · Canary ramp with a guardrail

In [ ]:
def canary_ramp(cand_name, cand_model, prod_model, traffic, steps=(0.05, 0.25, 0.5, 1.0), max_gap=0.05):
    rng = np.random.default_rng(0)
    order = rng.permutation(len(traffic))          # the same users join first at every step
    print(f"--- canary for {cand_name} ---")
    for share in steps:
        users = traffic.iloc[order[: max(1, int(share * len(traffic)))]]
        cand_rate = (cand_model.predict(users[FEATURES]) == "spam").mean()
        prod_rate = (prod_model.predict(users[FEATURES]) == "spam").mean()
        gap = abs(cand_rate - prod_rate)
        print(f"{share:>5.0%} ({len(users):>3} users): candidate flags {cand_rate:5.1%}, "
              f"production {prod_rate:5.1%}, gap {gap:4.1%}")
        if gap > max_gap:
            print(f"      guardrail broken -> ROLL BACK, only {len(users)} users ever saw {cand_name}")
            return False
    print(f"      guardrail held -> {cand_name} now serves everyone")
    return True


v4_survived = canary_ramp("v4", load_version("v4"), prod_model, traffic)
v2_survived = canary_ramp("v2", load_version("v2"), prod_model, traffic)

**A8.** **Shadow.** No user ever sees the candidate's answer, yet the report shows the problem: the candidate calls only 1.8% of messages spam, against 15.6% for production. That is a huge, label-free warning sign. **Canary** limits the damage: 45 users were routed to v4, and 4 of them got a wrong answer that production would have gotten right, compared with 66 for a big-bang switch. **Blue/green** gives the fastest rollback: one `switch()` and the old model is serving again.

## B8 · Monitoring: watch for drift

In [ ]:
reference = pd.read_csv(BASE_DATA)
DRIFT_THRESHOLD = 0.25   # a change of more than 25% counts as drift


def drift_report(batch, model):
    rows = []
    for feature in FEATURES:
        before, after = reference[feature].mean(), batch[feature].mean()
        rows.append({"check": f"average {feature}", "reference": before, "this week": after})
    before = (model.predict(reference[FEATURES]) == "spam").mean()
    after = (model.predict(batch[FEATURES]) == "spam").mean()
    rows.append({"check": "share predicted spam", "reference": before, "this week": after})

    report = pd.DataFrame(rows)
    report["change"] = (report["this week"] - report["reference"]) / report["reference"]
    report["drift"] = report["change"].abs() > DRIFT_THRESHOLD
    return report.round(3)


week_drift = pd.read_csv("data/new_week_drift.csv")
prod_model = joblib.load(PRODUCTION)
print("new_week_ok:")
display(drift_report(week_ok, prod_model))
print("new_week_drift:")
display(drift_report(week_drift, prod_model))

### 8b · See the drift

In [ ]:
bins = [0, 25, 50, 75, 100, 150, 200, 400]
ref_share = pd.cut(reference["length"], bins).value_counts(normalize=True).sort_index()
new_share = pd.cut(week_drift["length"], bins).value_counts(normalize=True).sort_index()

print(f"{'length':>12}   {'training data':<26} {'drift week':<26}")
for interval in ref_share.index:
    r, n = ref_share[interval], new_share[interval]
    print(f"{str(interval):>12}   {'█' * round(r * 40):<20}{r:4.0%}   {'█' * round(n * 40):<20}{n:4.0%}")

**Alarm thresholds:** the normal week changed at most about 8%, so a 25% threshold leaves room for noise without missing the drift week's 40–97% changes.

### 8c · When the labels arrive

In [ ]:
pd.DataFrame({
    "normal week": evaluate(prod_model, week_ok),
    "drift week": evaluate(prod_model, week_drift),
}).T[METRICS]

**A9.** All three: average length fell by about half, average punctuation by about 40%, and the share predicted spam collapsed from about 50% to about 1%. The **prediction share** is the best early alarm: it needs no labels, it is available minutes after the messages arrive, and it measures what users actually feel, which is spam no longer being caught. Once labels arrive, production's recall on the drift week is close to 0%: spammers now send short messages that look like ham to a model that only sees `length` and `punct`.

## B9 · Retrain after drift, and refresh the test set

In [ ]:
drift_train, drift_test = train_test_split(week_drift, test_size=0.5, random_state=42, stratify=week_drift[LABEL])
new_version = run_pipeline(drift_train, "drift_train", gate_strict)

The gate passed. How does the new model do on the new kind of spam?

In [ ]:
new_model = load_version(new_version)
pd.DataFrame({
    "fixed_test (old spam)": evaluate(new_model, fixed_test),
    "drift_test (new spam)": evaluate(new_model, drift_test),
}).T[METRICS]

### 9b · Refresh the test set

In [ ]:
refreshed_test = pd.concat([fixed_test, drift_test], ignore_index=True)

prod_model = joblib.load(PRODUCTION)
m = evaluate(prod_model, refreshed_test)
print(f"production = {production_version()} on refreshed test: accuracy {m['accuracy']:.1%}, recall {m['recall']:.1%}")
print("passes strict gate on the refreshed test set:", gate_strict(m))

### 9c · Keeping the test set fresh

In [ ]:
mix = pd.DataFrame({
    "messages": [len(fixed_test), len(drift_test)],
    "spam": [(fixed_test[LABEL] == "spam").sum(), (drift_test[LABEL] == "spam").sum()],
}, index=["old (fixed_test)", "new (drift_test)"])
mix["share of all spam"] = (mix["spam"] / mix["spam"].sum()).round(2)
mix

**A10.** 1. **The test set was out of date.** It only contained the old kind of spam, so the gate measured the wrong thing and gave a false PASS. A test set has to be refreshed with recent labeled data, just like the training data.
2. **The features are too weak.** The new spam (about 45 characters) and ham (about 50 characters) look almost the same in `length` and `punct`. No amount of retraining can separate them.

The numbers: the new model has 98.7% recall on the old spam but only 27.2% on the new spam, and 54% on the refreshed test set as a whole.

The real fix is **better features**: look at the words themselves, for example with TF-IDF or a deep learning text model. That is the bridge from this MLOps lab back to deep learning. Good operations cannot rescue a model that cannot see the signal.

## B10 · Close the loop

In [ ]:
def weekly_job(batch, name):
    print(f"\n##### Weekly job: {name} #####")
    report = drift_report(batch, joblib.load(PRODUCTION))
    if not report["drift"].any():
        print("No drift. Nothing to do.")
        return
    print("ALERT: drift in", ", ".join(report.loc[report["drift"], "check"]))
    check = validate_batch(batch, name)
    if not 0.3 <= check["spam_share"] <= 0.7:
        print("STOP: the label mix looks wrong. A human needs to check the labels.")
        return
    print("Label mix OK. Production disagreeing with the labels is expected during drift: the gate decides.")
    version = run_pipeline(batch, name, gate_strict, test_df=refreshed_test)
    if production_version() != version:
        print("Gate failed: a human needs to look at this (better features? more labels?)")


weekly_job(week_ok, "new_week_ok")
weekly_job(drift_train, "drift_train")

### The full history

In [ ]:
load_registry()[["version", "trained_on", "accuracy", "recall", "gate", "gate_result", "stage"]]

**A11.** v1 is the imbalanced model: high accuracy, useless recall, never promoted. v2 is the balanced model, the first one in production. v3 came from a normal week, passed and replaced v2. v4 came from the bad batch and was promoted by the accuracy-only gate, then rolled back. v5 is the same bad batch, blocked by the strict gate. v6 was retrained on the drift data and passed on the old test set. v7 was trained by the weekly job on the *same data* as v6, so it is the same model, but it was judged against the refreshed test set and failed (recall 54%). The only thing that changed between v6 and v7 is the test set. That shows how much a stale test set can hide. A human was alerted. **v6 stays in production.** It is the best model we have for the new traffic, and the system honestly reports that it is not good enough yet. That is a job for better features, not for more automation.

**A12.** Example: **GitHub Actions as a scheduler and CI/CD.** A workflow runs `weekly_job` whenever a new CSV is pushed to `data/`. If a model is promoted, it rebuilds and pushes the Docker image and starts a canary rollout. This would have run the bad batch through `validate_batch` and the strict gate automatically, and a failed run would show up as a red check on GitHub. Other good answers: DVC to version every weekly batch, Slack alerts for drift, MLflow to replace `registry.csv`, or SHAP to show *why* short spam is called ham.